In [25]:
!pip install gdown -q

In [26]:
import gdown

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [27]:
gdown.download(
    'https://drive.google.com/uc?id=1f1Rrn9xbWp7xDgcYYUSxuHHltQzqXUXK',
    'train_vitals.csv', quiet=False
)

gdown.download(
    'https://drive.google.com/uc?id=1QNxbxxt4LhuR3-BmdghaUz-XQ0tQhSC3',
    'test_vitals.csv', quiet=False
)

Downloading...
From: https://drive.google.com/uc?id=1f1Rrn9xbWp7xDgcYYUSxuHHltQzqXUXK
To: C:\Users\Admin\Desktop\Data Science\train_vitals.csv
100%|█████████████████████████████████████████████████████████████████████████████| 3.99M/3.99M [00:00<00:00, 4.02MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QNxbxxt4LhuR3-BmdghaUz-XQ0tQhSC3
To: C:\Users\Admin\Desktop\Data Science\test_vitals.csv
100%|█████████████████████████████████████████████████████████████████████████████| 17.8M/17.8M [00:05<00:00, 3.51MB/s]


'test_vitals.csv'

In [28]:
df_train = pd.read_csv('train_vitals.csv')
df_test = pd.read_csv('test_vitals.csv')

print('Train shape :', df_train.shape)
print('Test shape :', df_test.shape)
df_train.head()

Train shape : (122347, 6)
Test shape : (550583, 6)


,case_id,time_sec,HR,MBP,SpO2,Temp
0,135,1,112.0,-1.0,97.0,NaN
1,135,3,112.0,-1.0,97.0,NaN
2,135,5,111.0,-1.0,97.0,NaN
3,135,7,111.0,-2.0,97.0,NaN
4,135,9,112.0,-2.0,97.0,NaN


In [29]:
FEATURES = ['HR', 'MBP', 'SpO2', 'Temp']
df_train[FEATURES] = df_train[FEATURES].where(df_train[FEATURES] > 0)
df_test[FEATURES]  = df_test[FEATURES].where(df_test[FEATURES] > 0)

In [30]:
df_train[FEATURES] = df_train.groupby('case_id')[FEATURES].transform(
    lambda x: x.fillna(x.median())
)

df_test[FEATURES] = df_test.groupby('case_id')[FEATURES].transform(
    lambda x: x.fillna(x.median())
)

# fallback
df_train[FEATURES] = df_train[FEATURES].fillna(df_train[FEATURES].median())
df_test[FEATURES]  = df_test[FEATURES].fillna(df_test[FEATURES].median())

In [31]:
for col in FEATURES:
    df_train[f'{col}_mean'] = df_train.groupby('case_id')[col].transform(lambda x: x.rolling(5, min_periods=1).mean())
    df_test[f'{col}_mean']  = df_test.groupby('case_id')[col].transform(lambda x: x.rolling(5, min_periods=1).mean())

    df_train[f'{col}_std'] = df_train.groupby('case_id')[col].transform(lambda x: x.rolling(5, min_periods=1).std())
    df_test[f'{col}_std']  = df_test.groupby('case_id')[col].transform(lambda x: x.rolling(5, min_periods=1).std())

In [32]:
for col in FEATURES:
    df_train[f'{col}_diff'] = df_train.groupby('case_id')[col].diff().fillna(0)
    df_test[f'{col}_diff']  = df_test.groupby('case_id')[col].diff().fillna(0)



In [33]:
Thresholds = {
    'HR':   {'low': 40,   'high': 150},
    'MBP':  {'low': 50,   'high': 120},
    'SpO2': {'low': 90,   'high': 100},
    'Temp': {'low': 35.0, 'high': 39.5},
}

In [34]:
for col in FEATURES:
    low = Thresholds[col]['low']
    high = Thresholds[col]['high']

    df_train[f'{col}_abnormal'] = ((df_train[col] < low) | (df_train[col] > high)).astype(int)
    df_test[f'{col}_abnormal']  = ((df_test[col] < low) | (df_test[col] > high)).astype(int)

In [35]:
FEATURES_FINAL = [col for col in df_train.columns if col not in ['case_id', 'time_sec']]

In [36]:
X_train = df_train[FEATURES_FINAL]
X_test  = df_test[FEATURES_FINAL]

In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [38]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train)

,n_estimators,200
,max_samples,'auto'
,contamination,0.05
,max_features,1.0
,bootstrap,False
,n_jobs,-1
,random_state,42
,verbose,0
,warm_start,False


In [39]:
raw_scores = model.decision_function(X_test)

anomaly_scores = (raw_scores.max() - raw_scores) / (raw_scores.max() - raw_scores.min())

In [42]:
rule_score = df_test[[col+'_abnormal' for col in FEATURES]].sum(axis=1) / len(FEATURES)

final_score = 0.7 * anomaly_scores + 0.3 * rule_score
final_score

0         0.176611
1         0.180133
2         0.259798
3         0.239469
4         0.192840
            ...   
550578    0.284007
550579    0.273858
550580    0.267434
550581    0.265661
550582    0.295764
Length: 550583, dtype: float64

In [41]:
submission = pd.DataFrame({
    'case_id': df_test['case_id'],
    'time_sec': df_test['time_sec'],
    'anomaly_score': final_score
})

submission.to_csv('my_submission.csv', index=False)

In [45]:
submission.head()

,case_id,time_sec,anomaly_score
0,1,1,0.176611
1,1,3,0.180133
2,1,5,0.259798
3,1,7,0.239469
4,1,9,0.192840


In [48]:
submission.columns

Index(['case_id', 'time_sec', 'anomaly_score'], dtype='object')

In [49]:
submission.shape

(550583, 3)

In [50]:
submission.isnull().sum()

case_id          0
time_sec         0
anomaly_score    0
dtype: int64

In [51]:
submission['anomaly_score'].describe()

count    550583.000000
mean          0.096742
std           0.113033
min           0.000000
25%           0.025386
50%           0.049828
75%           0.126266
max           0.910842
Name: anomaly_score, dtype: float64

In [52]:
submission['anomaly_score'].min(), submission['anomaly_score'].max()

(0.0, 0.9108424463315616)

In [53]:
submission.sample(10)

,case_id,time_sec,anomaly_score
170336,41,2160,0.039260
498173,116,10304,0.178480
243602,56,18993,0.043559
453474,110,3947,0.025668
131562,28,15666,0.032977
309925,74,14349,0.012806
277772,65,8773,0.043705
524141,119,13421,0.047500
421268,98,5602,0.059703
545480,132,115,0.239108


In [54]:
assert submission.shape[0] == df_test.shape[0]
assert list(submission.columns) == ['case_id', 'time_sec', 'anomaly_score']
assert submission['anomaly_score'].between(0,1).all()

In [55]:
check_df = pd.read_csv('my_submission.csv')
check_df.head()

,case_id,time_sec,anomaly_score
0,1,1,0.176611
1,1,3,0.180133
2,1,5,0.259798
3,1,7,0.239469
4,1,9,0.192840


In [56]:
from IPython.display import FileLink

FileLink('my_submission.csv')

C:\Users\Admin\Desktop\Data Science\my_submission.csv

In [58]:
final_score = final_score ** 1.2
final_score

0         0.082358
1         0.084733
2         0.143574
3         0.127679
4         0.093473
            ...   
550578    0.163227
550579    0.154895
550580    0.149690
550581    0.148263
550582    0.173046
Length: 550583, dtype: float64

In [60]:
extreme = (
    (df_test['MBP'] < 40) |
    (df_test['SpO2'] < 85) |
    (df_test['HR'] > 180)
).astype(int)

final_score = np.maximum(final_score, extreme * 0.95)
final_score

0         0.082358
1         0.084733
2         0.143574
3         0.127679
4         0.093473
            ...   
550578    0.163227
550579    0.154895
550580    0.149690
550581    0.148263
550582    0.173046
Length: 550583, dtype: float64

In [61]:
rule_score = rule_score ** 0.8
rule_score

0         0.000000
1         0.000000
2         0.000000
3         0.000000
4         0.000000
            ...   
550578    0.329877
550579    0.329877
550580    0.329877
550581    0.329877
550582    0.329877
Length: 550583, dtype: float64

In [64]:
final_score = 0.75 * anomaly_scores + 0.25 * rule_score
final_score

0         0.189226
1         0.192999
2         0.278355
3         0.256574
4         0.206615
            ...   
550578    0.306405
550579    0.295532
550580    0.288649
550581    0.286749
550582    0.319002
Length: 550583, dtype: float64